# 🏦 Credit Risk Scoring Model — Indian Loan Applicants
**Author:** Hruthvik HS | **Tools:** Python, Scikit-learn, Pandas, Matplotlib
> ML-based credit risk system on 1,200 Indian loan applicants. Compares Logistic Regression, Random Forest, and Gradient Boosting. Identifies key default risk factors for Indian banks and NBFCs.

In [ ]:
!pip install scikit-learn pandas numpy matplotlib --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, accuracy_score)
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ Libraries loaded")

## 📊 Step 1 — Generate Indian Loan Applicant Dataset
Simulated based on realistic Indian lending market characteristics (CIBIL scores, income levels, loan norms).

In [ ]:
N = 1200
age            = np.random.randint(22, 62, N)
income_lpa     = np.round(np.random.lognormal(mean=np.log(6), sigma=0.6, size=N), 2)
loan_amt       = np.round(income_lpa * np.random.uniform(0.5, 4.0, N), 2)
loan_term      = np.random.choice([12,24,36,48,60], N, p=[0.1,0.2,0.35,0.25,0.1])
cibil_score    = np.clip(np.random.normal(720, 80, N).astype(int), 300, 900)
emp_years      = np.random.randint(0, 20, N)
existing_loans = np.random.choice([0,1,2,3], N, p=[0.4,0.35,0.18,0.07])
education      = np.random.choice(['Graduate','Post-Graduate','Under-Graduate','Diploma'], N, p=[0.45,0.25,0.2,0.1])
employment     = np.random.choice(['Salaried','Self-Employed','Business','Freelance'], N, p=[0.55,0.2,0.18,0.07])
property_area  = np.random.choice(['Urban','Semi-Urban','Rural'], N, p=[0.45,0.35,0.2])
gender         = np.random.choice(['Male','Female'], N, p=[0.62,0.38])
dependents     = np.random.choice([0,1,2,3], N, p=[0.35,0.3,0.25,0.1])

default_prob = (
    0.12 - 0.0003*(cibil_score-600) + 0.04*(loan_amt/(income_lpa+0.1)-2)
    + 0.02*existing_loans - 0.005*emp_years
    + 0.03*(employment=='Freelance').astype(int)
    + 0.02*(property_area=='Rural').astype(int)
    - 0.01*(education=='Post-Graduate').astype(int)
    + 0.02*dependents
)
default_prob = np.clip(default_prob, 0.04, 0.88)
default = (np.random.random(N) < default_prob).astype(int)

df = pd.DataFrame({
    'Age':age, 'Annual_Income_LPA':income_lpa, 'Loan_Amount_LPA':loan_amt,
    'Loan_Term_Months':loan_term, 'CIBIL_Score':cibil_score,
    'Employment_Years':emp_years, 'Existing_Loans':existing_loans,
    'Education':education, 'Employment_Type':employment,
    'Property_Area':property_area, 'Gender':gender,
    'Dependents':dependents, 'Default':default
})
df['Loan_to_Income_Ratio'] = (df['Loan_Amount_LPA'] / df['Annual_Income_LPA']).round(3)

print(f"✅ Dataset: {len(df)} applicants | Default rate: {df['Default'].mean()*100:.1f}%")
df.head()

## ⚙️ Step 2 — Feature Engineering & Model Training

In [ ]:
df_model = df.copy()
df_model['Is_Graduate'] = (df['Education'].isin(['Graduate','Post-Graduate'])).astype(int)
df_model['Is_Salaried'] = (df['Employment_Type']=='Salaried').astype(int)
df_model['Is_Urban']    = (df['Property_Area']=='Urban').astype(int)
df_model['Is_Female']   = (df['Gender']=='Female').astype(int)
df_model['CIBIL_Category'] = pd.cut(df['CIBIL_Score'], bins=[0,579,669,739,799,900],
                                     labels=[0,1,2,3,4]).astype(int)

features = ['Age','Annual_Income_LPA','Loan_Amount_LPA','Loan_Term_Months',
            'CIBIL_Score','Employment_Years','Existing_Loans','Dependents',
            'Loan_to_Income_Ratio','Is_Graduate','Is_Salaried','Is_Urban',
            'Is_Female','CIBIL_Category']

X = df_model[features]; y = df_model['Default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42),
}
results = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_tr_sc, y_train)
        y_pred  = model.predict(X_te_sc)
        y_proba = model.predict_proba(X_te_sc)[:,1]
    else:
        model.fit(X_train, y_train)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:,1]
    results[name] = {
        'model':model, 'pred':y_pred, 'proba':y_proba,
        'acc':round(accuracy_score(y_test,y_pred)*100,2),
        'auc':round(roc_auc_score(y_test,y_proba),4),
        'cm':confusion_matrix(y_test,y_pred),
        'report':classification_report(y_test,y_pred,output_dict=True),
    }
    print(f"  {name:25s} Accuracy: {results[name]['acc']}%  |  AUC: {results[name]['auc']}")

best_name = max(results, key=lambda k: results[k]['auc'])
print(f"\n🏆 Best Model: {best_name} (AUC={results[best_name]['auc']})")

## 🎨 Step 3 — Visualise Results

In [ ]:
BG='#0A0E1A'; PANEL='#111827'; TEXT='#F0F4FF'; SUB='#8892A4'
ACC='#6C63FF'; GREEN='#00D97E'; RED='#FF4560'; GOLD='#FFB400'; TEAL='#00C9C9'

fig, axes = plt.subplots(2, 3, figsize=(18, 11), facecolor=BG)
fig.suptitle('CREDIT RISK SCORING MODEL — INDIAN LOAN APPLICANTS',
             fontsize=16, fontweight='bold', color=TEXT, fontfamily='monospace')

colors_roc = [ACC, GREEN, GOLD]
ax1 = axes[0,0]; ax1.set_facecolor(PANEL)
for (name, res), col in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    ax1.plot(fpr, tpr, color=col, linewidth=2, label=f"{name} ({res['auc']})")
ax1.plot([0,1],[0,1], color=SUB, linewidth=1, linestyle='--', alpha=0.5)
ax1.set_title('ROC Curves', color=TEXT, fontsize=11, fontweight='bold')
ax1.legend(facecolor='#1A2235', edgecolor='#2D3748', labelcolor=TEXT, fontsize=7)
ax1.tick_params(colors=SUB); ax1.set_xlabel('FPR', color=SUB); ax1.set_ylabel('TPR', color=SUB)
for sp in ax1.spines.values(): sp.set_color('#2D3748')
ax1.grid(alpha=0.08, color=SUB)

ax2 = axes[0,1]; ax2.set_facecolor(PANEL)
cm = results[best_name]['cm']
ax2.imshow(cm, cmap='Blues', aspect='auto')
for i in range(2):
    for j in range(2):
        ax2.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=20,fontweight='bold',
                 color='white' if cm[i,j]>cm.max()*0.5 else TEXT)
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(['Pred: No Default','Pred: Default'],color=TEXT,fontsize=8)
ax2.set_yticklabels(['Actual: No Default','Actual: Default'],color=TEXT,fontsize=8)
ax2.set_title(f'Confusion Matrix\n{best_name}',color=TEXT,fontsize=11,fontweight='bold')
for sp in ax2.spines.values(): sp.set_color('#2D3748')

rf_model = results['Random Forest']['model']
fi = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)
ax3 = axes[0,2]; ax3.set_facecolor(PANEL)
fi_colors = [GREEN if v>fi.median() else TEAL for v in fi.values]
ax3.barh(fi.index, fi.values, color=fi_colors, height=0.65, edgecolor='none')
ax3.set_title('Feature Importance
(Random Forest)', color=TEXT, fontsize=11, fontweight='bold')
ax3.tick_params(colors=SUB, labelsize=8)
for sp in ax3.spines.values(): sp.set_color('#2D3748')
ax3.grid(axis='x', alpha=0.1, color=SUB)

model_names = list(results.keys())
accs = [results[m]['acc'] for m in model_names]
aucs = [results[m]['auc']*100 for m in model_names]
x_pos = np.arange(len(model_names)); w = 0.35
ax4 = axes[1,0]; ax4.set_facecolor(PANEL)
b1 = ax4.bar(x_pos-w/2, accs, width=w, color=ACC,  label='Accuracy (%)', edgecolor='none')
b2 = ax4.bar(x_pos+w/2, aucs, width=w, color=GOLD, label='AUC×100',      edgecolor='none')
for bar,val in zip(list(b1)+list(b2), accs+aucs):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             f'{val:.1f}', ha='center', fontsize=8, color=TEXT, fontweight='bold')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(['Logistic\nRegression','Random\nForest','Gradient\nBoosting'],color=TEXT,fontsize=8)
ax4.set_title('Model Comparison', color=TEXT, fontsize=11, fontweight='bold')
ax4.legend(facecolor='#1A2235', edgecolor='#2D3748', labelcolor=TEXT, fontsize=8)
ax4.tick_params(colors=SUB); ax4.set_ylim(0,105)
for sp in ax4.spines.values(): sp.set_color('#2D3748')
ax4.grid(axis='y', alpha=0.1, color=SUB)

bins_c = [300,500,580,670,740,800,900]
lbls_c = ['300-500','500-580','580-670','670-740','740-800','800+']
df['CIBIL_Band'] = pd.cut(df['CIBIL_Score'],bins=bins_c,labels=lbls_c)
cibil_def = df.groupby('CIBIL_Band',observed=True)['Default'].mean()*100
ax5 = axes[1,1]; ax5.set_facecolor(PANEL)
bc5 = [RED if v>30 else GOLD if v>15 else GREEN for v in cibil_def.values]
bars5 = ax5.bar(range(len(cibil_def)), cibil_def.values, color=bc5, width=0.6, edgecolor='none')
ax5.set_xticks(range(len(cibil_def))); ax5.set_xticklabels(lbls_c,color=SUB,fontsize=8,rotation=20)
for bar,val in zip(bars5,cibil_def.values):
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.1f}%', ha='center', fontsize=9, color=TEXT, fontweight='bold')
ax5.set_title('Default Rate by CIBIL Band', color=TEXT, fontsize=11, fontweight='bold')
ax5.tick_params(colors=SUB); ax5.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
for sp in ax5.spines.values(): sp.set_color('#2D3748')
ax5.grid(axis='y', alpha=0.1, color=SUB)

lti_bins = [0,1,1.5,2,2.5,3,10]; lti_lbls = ['<1x','1-1.5x','1.5-2x','2-2.5x','2.5-3x','>3x']
df['LTI_Band'] = pd.cut(df['Loan_to_Income_Ratio'],bins=lti_bins,labels=lti_lbls)
lti_def = df.groupby('LTI_Band',observed=True)['Default'].mean()*100
ax6 = axes[1,2]; ax6.set_facecolor(PANEL)
bc6 = [RED if v>35 else GOLD if v>20 else GREEN for v in lti_def.values]
bars6 = ax6.bar(range(len(lti_def)), lti_def.values, color=bc6, width=0.6, edgecolor='none')
ax6.set_xticks(range(len(lti_def))); ax6.set_xticklabels(lti_lbls,color=SUB,fontsize=9)
for bar,val in zip(bars6,lti_def.values):
    ax6.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.1f}%', ha='center', fontsize=9, color=TEXT, fontweight='bold')
ax6.set_title('Default Rate by Loan-to-Income Ratio', color=TEXT, fontsize=11, fontweight='bold')
ax6.tick_params(colors=SUB); ax6.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
for sp in ax6.spines.values(): sp.set_color('#2D3748')
ax6.grid(axis='y', alpha=0.1, color=SUB)

plt.tight_layout()
plt.savefig('credit_risk_dashboard.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ Dashboard generated!")

## 💡 Step 4 — Business Insights

In [ ]:
best = results[best_name]
rep  = best['report']
print("=" * 55)
print("       CREDIT RISK MODEL — KEY FINDINGS")
print("=" * 55)
print(f"  Best Model       : {best_name}")
print(f"  Accuracy         : {best['acc']}%")
print(f"  AUC-ROC          : {best['auc']}")
print(f"  Precision (Default): {rep['1']['precision']:.2f}")
print(f"  Recall (Default)   : {rep['1']['recall']:.2f}")
print(f"  F1 Score (Default) : {rep['1']['f1-score']:.2f}")
print("=" * 55)
print("\n  TOP RISK FACTORS (Random Forest):")
top_f = pd.Series(rf_model.feature_importances_, index=features).nlargest(5)
for feat, imp in top_f.items():
    print(f"  {feat:30s} {imp:.4f}")
print("\n  BUSINESS RECOMMENDATIONS:")
print("  1. Reject applicants with CIBIL < 580 (default rate >30%)")
print("  2. Cap loans at 2x annual income to limit risk")
print("  3. Flag freelancers and applicants with 3+ existing loans")
print("  4. Urban salaried applicants show lowest default risk")